In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from lightgbm import LGBMRegressor

# =========================
# Config
# =========================
N_SPLITS = 5
RANDOM_STATE = 42

# =========================
# Load data
# =========================
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

y = np.log1p(train_df["SalePrice"])
train_df = train_df.drop(columns=["SalePrice"])

all_df = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)

# =========================
# Minimal preprocessing
# =========================

# Numerical
num_cols = all_df.select_dtypes(include=["int64", "float64"]).columns
all_df[num_cols] = all_df[num_cols].fillna(all_df[num_cols].median())

# Categorical
cat_cols = all_df.select_dtypes(include=["object"]).columns
all_df[cat_cols] = all_df[cat_cols].fillna("Missing")

# One-hot encoding
all_df = pd.get_dummies(all_df, columns=cat_cols, drop_first=True)

X = all_df.iloc[:len(train_df)]
X_test = all_df.iloc[len(train_df):]

# =========================
# Cross Validation
# =========================
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    print(f"Fold {fold + 1}")

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = LGBMRegressor(
        n_estimators=2000,
        learning_rate=0.03,
        max_depth=-1,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        eval_metric="rmse",
        verbose=False,
        early_stopping_rounds=100,
    )

    oof_preds[val_idx] = model.predict(X_val)
    test_preds += model.predict(X_test) / N_SPLITS

rmse = mean_squared_error(y, oof_preds, squared=False)
print(f"CV RMSE: {rmse:.5f}")

# =========================
# Submission
# =========================
submission = pd.DataFrame({
    "Id": pd.read_csv("test.csv")["Id"],
    "SalePrice": np.expm1(test_preds)
})

submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")
